In [ ]:
import pandas as pd

### Types of Recommendation Systems<br>
1> Content based <br>
2> Collaborating filtering <br>
3> Hybrid

In [ ]:
movies = pd.read_csv("D://sampledata//Projects//Movie_Recommender//tmdb_5000_movies.csv")
credits = pd.read_csv("D://sampledata//Projects//Movie_Recommender//tmdb_5000_credits.csv")

In [ ]:
movies.head()

In [ ]:
credits.head()

In [ ]:
movies.merge(credits,on='title').shape

In [ ]:
movies.shape, credits.shape

In [ ]:
movies = movies.merge(credits,on='title')

In [ ]:
movies.head(1)

In [ ]:
movies.columns

### List of Columns needed for this project
genres, id, keywords, title, overview, cast, crew

In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.head(1)

## Preprocessing

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

['Action', 'Adventure', 'Fantasy', 'Science Fiction' ]

In [ ]:
# genres has a string data type, it is needed to convert it into list
import ast
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
movies['keywords'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
movies['cast'][0]

In [ ]:
def convert2(obj):
    L = []
    count = 0 
    for i in ast.literal_eval(obj):
        if count != 3:
            L.append(i['name'])
            count += 1
        else:
            break
    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert2)

In [ ]:
movies.head(3)

In [ ]:
movies['crew'][0]

In [ ]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head(3)

In [ ]:
movies['overview'][0]

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies.head(3)

In [ ]:
columns = ['genres','keywords','cast','crew']

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies.head(4)

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head(2)

In [ ]:
new_df = movies[['movie_id', 'title','tags']]

In [ ]:
new_df

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [ ]:
new_df['tags'][0]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [ ]:
new_df['tags'][0]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=500, stop_words='english')

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors.shape

In [ ]:
cv.get_feature_names_out()

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [ ]:
stem('in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron')

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
from sklearn.metrics.pairwise import  cosine_similarity

In [ ]:
cosine_similarity(vectors).shape

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity[0]

In [ ]:
new_df['title'] == 'Avatar'

In [ ]:
new_df[new_df['title'] == 'Avatar']

In [ ]:
new_df[new_df['title'] == 'Avatar'].index

In [ ]:
new_df[new_df['title'] == 'Batman Begins'].index[0]

In [ ]:
sorted(similarity[0], reverse=True)

In [ ]:
list(enumerate(similarity[0]))

In [ ]:
sorted(list(enumerate(similarity[0])))

In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True, key= lambda x:x[1])[1:6]

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True, key= lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Batman Begins')

In [ ]:
recommend('Avatar')

In [68]:
import pickle as pk

In [70]:
pk.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))

In [71]:
pk.dump(similarity,open('similarity.pkl','wb'))